In [21]:
# Some helper functions
import re
import io
import itertools as itr
from contextlib import redirect_stdout

def get_prop_of_lie_algebra(letter,rnk):
    ''' 
    Function to compute the dimension and dual coxeter number of a Lie Algebra
    Input: letter (string) - cartan label of the lie algbra
            rnk (int) - rank of the lie algebra
    Output: dim (int) - dimension of the lie Algebra
            dcox (int) - dual Coxeter number
    '''
    # Default values indicate errors
    dim = -1 # Dimension of the Lie Algebra
    dcox = -1 # Rank of the Lie Algebra
    hr_vector = -1 # The highest root vector
    quadF = -1 # The quadratic form
    # Go through all the Cartan types
    if letter == 'A':
        dim = rnk**2 + 2*rnk
        dcox = rnk + 1
        if rnk >= 2:
            hr_vector = vector(ZZ, [1]+(rnk-2)*[0]+[1])
        elif rnk == 1:
            hr_vector = vector(ZZ,[1])
        
    elif letter == 'B':
        dim = 2*rnk**2 + rnk
        dcox = 2*rnk - 1
        hr_vector = vector(ZZ, [0,1]+(rnk-2)*[0])
    elif letter == 'C':
        dim = 2*rnk**2 + rnk
        dcox = rnk + 1
        hr_vector = vector(ZZ, [2]+[0]*(rnk-1))
    elif letter == 'D':
        dim = 2*rnk^2 - rnk
        dcox = 2*rnk - 2
        hr_vector = vector(ZZ, [0,1]+(rnk-2)*[0])
    elif letter == 'E':
        if rnk == 8:
            dim = 248
            dcox = 30
            hr_vector = vector(ZZ, [1]+(rnk-1)*[0])
        elif rnk == 7:
            dim = 133
            dcox = 18
            hr_vector = vector(ZZ, [1]+(rnk-1)*[0])
        elif rnk == 6:
            dim = 78
            dcox = 12
            hr_vector = vector(ZZ, (rnk-1)*[0]+[1])
    elif letter == 'F':
        if rnk == 4:
            dim = 52
            dcox = 9
            hr_vector = vector(ZZ, [1]+(rnk-1)*[0])
    elif letter == 'G':
        if rnk == 2:
            dim = 14
            dcox = 4
            hr_vector = vector(ZZ, [1,0])
    
    return dim,dcox


def weight_projection(b_rule):
    '''
    Function to get the weight lattice projection matrix from the branching rule.
    Input: b_rule (branching rule object) - The branching rule in question
    Output: proj_matrices (list) - 
    '''
    # Get the description as a string
    f = io.StringIO()
    with redirect_stdout(f):
        b_rule.describe(verbose=True)
    b_d = f.getvalue().strip()
    # Now build the dictionary by looping through the lines of the description
    # We extract the relevant information about fundamental weight restrictions
    weight_section = False
    proj_matrices = [] 
    for line in b_d.split('\n'):
        if weight_section and line.strip()=="":
            weight_section=False
            proj_matrices.append((subalg, matrix(QQ,proj_mat).T))
        if weight_section:
            proj_mat += [eval(line.split(" => ")[1].replace(' ',''))]
        if "fundamental weight restrictions" in line:
            weight_section = True
            subalg = line.split(" => ")[1].strip()[:2]
            proj_mat = []

    return proj_matrices


def correct_branching(b_rule):
    '''Function that corrects some erreneous branching rules that are bugs in SageMath'''

    if b_rule == branching_rule("B5","D2xB3","extended")*branching_rule("D2xB3","A1xA2xB3",[branching_rule("D2","A1xA1","isomorphic"),"identity"]):
        return branching_rule("B5","D2xB3","extended")*branching_rule("D2xB3","A1xA1xB3",[branching_rule("D2","A1xA1","isomorphic"),"identity"])
    elif b_rule == branching_rule("C4","C1xC3","orthogonal_sum")*branching_rule("C1xC3","A1xA3",[branching_rule("C1","A1","isomorphic"),"identity"]):
        return branching_rule("C4","C1xC3","orthogonal_sum")*branching_rule("C1xC3","A1xC3",[branching_rule("C1","A1","isomorphic"),"identity"])
    elif b_rule == branching_rule("D5","B1xB3","orthogonal_sum")*branching_rule("B1xB3","A1xA3",[branching_rule("B1","A1","isomorphic"),"identity"]):
        return branching_rule("D5","B1xB3","orthogonal_sum")*branching_rule("B1xB3","A1xB3",[branching_rule("B1","A1","isomorphic"),"identity"])
    elif b_rule == branching_rule("D6","C1xC3","tensor")*branching_rule("C1xC3","A1xA3",[branching_rule("C1","A1","isomorphic"),"identity"]):
        return branching_rule("D6","C1xC3","tensor")*branching_rule("C1xC3","A1xC3",[branching_rule("C1","A1","isomorphic"),"identity"])
    
    else:
        return b_rule

def find_subalg(cartan_type, nested = True):
    '''
    Function to find all the subalgebras of a given simple Lie algebra of rank less than eight.
    Input: The cartan type of a simple Lie algebra as a string (e.g. "D7")
    Output: all_b_rules - list of pairs; first element is a branching rule second the number of possible u(1) factors
    '''

    # Cast the cartan type and get the rank
    cartan_type = CartanType(cartan_type)
    rnk = cartan_type.rank()

    # Output list to fill
    all_b_rules = []

    # These simple base cases will break the code that follows so are implemented explicitly
    if rnk == 1:
        return all_b_rules
    elif cartan_type == CartanType("B2"):
        iso_b_rule = branching_rule("B2","C2","isomorphic")
        all_b_rules.append(( iso_b_rule*branching_rule("C2","A1","symmetric_power"), 0 ))
        all_b_rules.append(( iso_b_rule*branching_rule("C2","C1xC1","orthogonal_sum")*branching_rule("C1xC1","A1xA1",[branching_rule("C1","A1","isomorphic"),branching_rule("C1","A1","isomorphic")]),0 ))
        return all_b_rules

    # Get the Weyl character ring
    our_weyl_ring = WeylCharacterRing(cartan_type)

    # Now we get a list of all maximal subgroups 
    f = io.StringIO()
    with redirect_stdout(f):
        our_weyl_ring.maximal_subgroups()
    output = f.getvalue().strip()
    # Parse the text 
    for line in output.split('\n'):
        if not line.strip(): 
            continue
        # Sage prints things like: C2:branching_rule("A3","C2","symmetric")
        # We split at the first colon to isolate the cartan type and command
        command_str = line.split(':', 1)[1]
        # Evaluate the string to turn it into a real SageMath object.
        b_rule = eval(command_str)
        # Unfortunately, sagemath has some minor errors in it's branching rules which are corrected here
        b_rule = correct_branching(b_rule)

        # Get the cartan types and ranks
        all_ct_subs = b_rule.Stype()
        if 'x' in str(all_ct_subs):
            list_ct_subs = str(all_ct_subs).split('x')
            list_ct_subs = [ CartanType(ct_sub) for ct_sub in list_ct_subs ]
        else:
            list_ct_subs = [all_ct_subs]
        num_poss_u1 = rnk - sum( ct_sub.rank() for ct_sub in list_ct_subs )
        
        # If nested is False, look only at the maximal subgroups
        if not nested:
            all_b_rules.append( (b_rule,num_poss_u1) )
            continue

        # For each factor in str_ct_sub we see if there are further branchings possible
        further_branches = [find_subalg(ct_sub)+[(branching_rule(ct_sub,ct_sub,"identity"),0)] for ct_sub in list_ct_subs]
        further_b_rules = [ [correct_branching(p[0]) for p in bct_sub] for bct_sub in further_branches ]
        further_u1_adds = [ [p[1] for p in bct_sub] for bct_sub in further_branches ]
        all_branch_ind = [ range(len(b)) for b in further_b_rules ]
        begin_type = b_rule.Stype()
        b_rule_added = False
        for branch_indices in itr.product(*all_branch_ind):
            these_b_rules = [ further_b_rules[i][branch_indices[i]] for i in range(len(further_b_rules))]
            if all(str(b).split(" ")[0] == "identity" for b in these_b_rules):
                if not b_rule_added:
                    all_b_rules.append( (b_rule, num_poss_u1) )
                b_rule_added = True
                continue
            end_type = ''
            for b in these_b_rules:
                this_ct = b.Stype()
                if this_ct.is_irreducible():
                    end_type += this_ct.type() + str(this_ct.rank()) + 'x'
                else:
                    end_type += str(this_ct) + 'x'
            end_type = end_type[:-1]
            total_u1 = num_poss_u1 + sum( further_u1_adds[i][branch_indices[i]] for i in range(len(further_u1_adds)) )
            if len(these_b_rules) == 1:
                all_b_rules.append( ( b_rule*these_b_rules[0], total_u1 ) )
            else:
                all_b_rules.append( ( b_rule*branching_rule(begin_type,end_type,these_b_rules), total_u1 ) )
    return all_b_rules 



In [ ]:
import io
import itertools as itr

target_c = 7/5

for letter in ['A','B','C','D','E','F','G']:
    if letter == 'E':
        rank_range = [6,7,8]
    elif letter == 'F':
        rank_range = [4]
    elif letter == 'G':
        rank_range = [2]
    elif letter == 'B':
        rank_range = srange(3,9)
    elif letter == 'C':
        rank_range = srange(2,9)
    elif letter == 'D':
        rank_range = srange(4,9)
    elif letter == 'A':
        rank_range = srange(2,9)
    for rnk in rank_range:
        dim,dcox = get_prop_of_lie_algebra(letter,rnk)
        cartan_type = CartanType([letter,rnk])
        all_b_rules = find_subalg(cartan_type, nested=True)
        coroot_form = CartanMatrix(cartan_type).dual().symmetrized_matrix()
        seen_embeddings = set()
        for b_rule,max_num_u1 in all_b_rules:
            # Get the projection matrices of the weight lattice
            try: 
                proj_matrices = weight_projection(b_rule)
            except:
                print(b_rule)
                continue
            # Loop through all the simple embedded algebras
            dynk_indices = []
            emb_algs = []
            emb_dims = []
            emb_dcoxes = []
            for emb_alg,proj_mat in proj_matrices:
                emb_rnk = ZZ( emb_alg[1:] )
                # Build the quadratic form
                emb_crt_form = CartanMatrix(emb_alg).dual().symmetrized_matrix()
                proj_crt_form = proj_mat * coroot_form * proj_mat.T
                dynk_ind = proj_crt_form[0,0] / emb_crt_form[0,0]
                if proj_crt_form != dynk_ind * emb_crt_form or dynk_ind.denominator() != 1:
                    raise RuntimeError(f"There is no well-defined, integral embedding index for the branching rule {b_rule}!") 
                dynk_indices.append(dynk_ind)
                emb_algs.append(emb_alg)
                ed,dc = get_prop_of_lie_algebra(emb_alg[0],emb_rnk)
                emb_dims.append(ed)
                emb_dcoxes.append(dc)
            
            # Now we see whether we can hit our target
            for num_u1 in range(max_num_u1+1):  
                this_embedding = (tuple(emb_algs), tuple(dynk_indices), num_u1)
                if this_embedding in seen_embeddings:
                    continue
                seen_embeddings.add( this_embedding )
                Polyk = PolynomialRing(ZZ,'k')
                k = Polyk.gen()
                denom = target_c.denominator()
                our_eqn = Polyk( (-(target_c+num_u1)*(k+dcox)+k*dim)*denom*product(dynk_indices[i]*k+emb_dcoxes[i] for i in range(len(proj_matrices))) )
                our_eqn += -denom*k*(k+dcox)*sum(dynk_indices[i]*emb_dims[i]*product(dynk_indices[j]*k+emb_dcoxes[j] for j in range(len(proj_matrices)) if j!=i) for i in range(len(proj_matrices)))
                for poss_k in our_eqn.roots(ring=ZZ):
                    if poss_k[0]>0:
                        print(b_rule)
                        print(poss_k[0])
                        print(dynk_indices)
                        print(num_u1)
                        print('\n')

composite branching rule A7 => (symmetric) C4 => (tensor) C1xD2 => ([isomorphic branching rule C1 => A1, isomorphic branching rule D2 => A1xA1]) A1xA1xA1
2
[4, 4, 4]
4


composite branching rule A7 => (tensor) A1xA3 => ([identity branching rule A1 => A1, tensor branching rule A3 => A1xA1]) A1xA1xA1
2
[4, 4, 4]
4


composite branching rule A8 => (tensor) A2xA2 => ([levi branching rule A2 => A1, levi branching rule A2 => A1]) A1xA1
1
[3, 3]
3


composite branching rule A8 => (levi) A1xA6 => ([identity branching rule A1 => A1, composite branching rule A6 => (symmetric) B3 => (miscellaneous) G2]) A1xG2
3
[1, 2]
0


composite branching rule B6 => (orthogonal_sum) B1xD5 => ([isomorphic branching rule B1 => A1, 'identity']) A1xD5 => ([identity branching rule A1 => A1, composite branching rule D5 => (plethysm (along C2(2,0))) C2 => (orthogonal_sum) C1xC1 => ([isomorphic branching rule C1 => A1, isomorphic branching rule C1 => A1]) A1xA1]) A1xA1xA1
1
[2, 3, 3]
0


composite branching rule B7 =>

In [22]:
WeylCharacterRing("B6").maximal_subgroups()

D6:branching_rule("B6","D6","extended")
A1:branching_rule("B6","A1","symmetric_power")
A1xA1xB4:branching_rule("B6","D2xB4","orthogonal_sum")*branching_rule("D2xB4","A1xA1xB4",[branching_rule("D2","A1xA1","isomorphic"),"identity"])
A1xD5:branching_rule("B6","B1xD5","orthogonal_sum")*branching_rule("B1xD5","A1xD5",[branching_rule("B1","A1","isomorphic"),"identity"])
A3xB3:branching_rule("B6","D3xB3","orthogonal_sum")*branching_rule("D3xB3","A3xB3",[branching_rule("D3","A3","isomorphic"),"identity"])
B2xD4:branching_rule("B6","B2xD4","orthogonal_sum")


In [25]:
simple_coroots = weight_lattice.simple_coroots()
coroot_form = matrix(QQ, [[simple_coroots[i].to_ambient().inner_product(simple_coroots[j].to_ambient()) for i in range(1,rnk+1)] for j in range(1,rnk+1)])


In [ ]:
b_rule = branching_rule("B5","D2xB3","extended")*branching_rule("D2xB3","A1xA2xB3",[branching_rule("D2","A1xA1","isomorphic"),"identity"])
b_rule.describe(verbose=True)


    O 0
    |
    |
O---O---O---O=>=O
1   2   3   4   5   
B5~
root restrictions B5 => A1:

O
1   
A1

0 => (zero)
1 => 1
2 => weight (-1/2, 1/2)
3 => (zero)
4 => (zero)
5 => (zero)

fundamental weight restrictions B5 => A1:
1 => (1,)
2 => (0,)
3 => (0,)
4 => (0,)
5 => (0,)

projection 1 on A1  None
root restrictions B5 => A2:

O---O
1   2   
A2

0 => root (-1, 1, 0)
1 => (zero)
2 => weight (1/2, -1/2, -1)
3 => weight (0, 0, 1)
4 => (zero)
5 => (zero)

fundamental weight restrictions B5 => A2:
1 => (1, -1/2)
2 => (2, -1)
3 => (2, -2)
4 => (2, -2)
5 => (1, -1)

projection 2 on A2  None
root restrictions B5 => B3:

O---O=>=O
1   2   3   
B3

0 => (zero)
1 => (zero)
2 => (zero)
3 => root (-1, 0, 0)
4 => 1
5 => root (0, 1, 0)

fundamental weight restrictions B5 => B3:
1 => (0, 0, 0)
2 => (0, 0, 0)
3 => (0, 0, 0)
4 => (1, 0, 0)
5 => (0, 1/2, 0)

projection 3 on B3  None


In [ ]:
F = matrix([[2,2,2,1],[2,4,4,2],[2,4,6,3],[1,2,3,2]])
P = matrix([[1,2,2,1]])
P*F*P.T

[114]

In [ ]:
emb_rnk = 3
emb_inv_cartan_mat = CartanMatrix('B3').inverse()
emb_wl = RootSystem('B3').weight_lattice()
emb_simp_rts = emb_wl.simple_roots()
emb_rt_len = emb_wl.highest_root().norm_squared()
emb_quadF = matrix(QQ, [[emb_inv_cartan_mat[i,j]*emb_simp_rts[i+1].norm_squared()/emb_rt_len for j in range(emb_rnk)] for i in range(emb_rnk)] )
show(emb_quadF)

[  1   1 1/2]
[  1   2   1]
[1/2   1 3/4]

In [ ]:
b_rule = branching_rule("B4","D2xB2","extended")*branching_rule("D2xB2","A1xA1xB2",[branching_rule("D2","A1xA1","isomorphic"),"identity"])
print(b_rule)
b_rule.describe(verbose=True)

In [ ]:
WeylCharacterRing("B6").maximal_subgroups()

In [16]:
(branching_rule("B6","D2xB4","orthogonal_sum")*branching_rule("D2xB4","A1xA1xB4",[branching_rule("D2","A1xA1","isomorphic"),"identity"])).describe(verbose=True)


    O 0
    |
    |
O---O---O---O---O=>=O
1   2   3   4   5   6   
B6~
root restrictions B6 => A1:

O
1   
A1

0 => (zero)
1 => 1
2 => weight (-1/2, 1/2)
3 => (zero)
4 => (zero)
5 => (zero)
6 => (zero)

fundamental weight restrictions B6 => A1:
1 => (1,)
2 => (0,)
3 => (0,)
4 => (0,)
5 => (0,)
6 => (0,)

projection 1 on A1  None
root restrictions B6 => A1:

O
1   
A1

0 => root (-1, 1)
1 => (zero)
2 => weight (1/2, -1/2)
3 => (zero)
4 => (zero)
5 => (zero)
6 => (zero)

fundamental weight restrictions B6 => A1:
1 => (1,)
2 => (2,)
3 => (2,)
4 => (2,)
5 => (2,)
6 => (1,)

projection 2 on A1  None
root restrictions B6 => B4:

O---O---O=>=O
1   2   3   4   
B4

0 => (zero)
1 => (zero)
2 => root (-1, 0, 0, 0)
3 => 1
4 => 2
5 => 3
6 => 4

fundamental weight restrictions B6 => B4:
1 => (0, 0, 0, 0)
2 => (0, 0, 0, 0)
3 => (1, 0, 0, 0)
4 => (0, 1, 0, 0)
5 => (0, 0, 1, 0)
6 => (0, 0, 0, 1)

projection 3 on B4  None


In [ ]:

import io
import re
import contextlib

def _parse_coords(s):
    return [QQ(t.strip()) for t in s.split(",")]

def _image_from_rhs(rhs, H):
    """
    Parse the RHS of a root-restriction line in b.describe(verbose=True).

    Accepted forms in the ROOT-RESTRICTIONS section:
      1
      root (-1, -1, 0)
      weight (-2, 0)
      (1, 0)        # defensive fallback, though this usually appears only
                    # in the fundamental-weight section
    """
    rhs = rhs.strip()

    # simple root label
    if re.fullmatch(r"\d+", rhs):
        j = int(rhs)
        return H.ambient_space().simple_roots()[j]

    # root (...) or weight (...)
    m = re.match(r"^(root|weight)\s*\(([^()]*)\)\s*$", rhs)
    if m:
        kind = m.group(1)
        coords = _parse_coords(m.group(2))
        if kind == "root":
            return H.ambient_space()(coords)
        else:
            # Use the fundamental weights to realize a weight-coordinate tuple
            fw = H.ambient_space().fundamental_weights()
            return sum(coords[i-1] * fw[i] for i in range(1, len(coords)+1))

    # bare tuple fallback
    m = re.match(r"^\(([^()]*)\)\s*$", rhs)
    if m:
        coords = _parse_coords(m.group(1))
        fw = H.ambient_space().fundamental_weights()
        return sum(coords[i-1] * fw[i] for i in range(1, len(coords)+1))

    raise ValueError(f"Unrecognized restriction image: {rhs!r}")


def projected_highest_root_from_branching_rule(b, source_type=None, target_type=None):
    """
    Image of the highest root of the source under branching rule b,
    computed from the root-restriction data only.
    """
    if source_type is None or target_type is None:
        # Parse from string form like "symmetric branching rule A4 => B2"
        s = str(b)
        m = re.search(r"branching rule\s+(.+?)\s*=>\s*(.+)$", s)
        if not m:
            raise ValueError(f"Could not parse source/target types from {s!r}")
        source_type = source_type or m.group(1).strip()
        target_type = target_type or m.group(2).strip()

    G = RootSystem(source_type)
    H = RootSystem(target_type)

    G_amb = G.ambient_space()
    H_amb = H.ambient_space()

    # Highest root of the source
    theta_G = G_amb.highest_root()

    # Express theta_G in the source simple-root basis
    sroots = G_amb.simple_roots()
    idx = sorted(sroots.keys())
    M = matrix(QQ, [vector(QQ, sroots[i]) for i in idx]).transpose()
    coeffs = M.solve_right(vector(QQ, theta_G))

    # Read only the ROOT-RESTRICTIONS block
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        b.describe(verbose=True)
    txt = buf.getvalue()

    in_root_block = False
    images = {}

    for line in txt.splitlines():
        if "root restrictions" in line:
            in_root_block = True
            continue
        if in_root_block and "fundamental weight restrictions" in line:
            break
        if not in_root_block:
            continue

        m = re.match(r"\s*(\d+)\s*=>\s*(.+?)\s*$", line)
        if not m:
            continue

        i = int(m.group(1))
        rhs = m.group(2).strip()

        # We only need the finite simple roots here; the affine root 0 is optional.
        if i == 0:
            continue

        images[i] = _image_from_rhs(rhs, H)

    proj = H_amb.zero()
    for i, c in zip(idx, coeffs):
        if c != 0:
            if i not in images:
                raise ValueError(f"Missing restriction data for simple root {i}")
            proj += c * images[i]

    return proj, H_amb.highest_root()


def embedding_index_from_branching_rule(b, source_type=None, target_type=None):
    proj_theta, theta_H = projected_highest_root_from_branching_rule(
        b, source_type=source_type, target_type=target_type
    )
    return proj_theta.inner_product(proj_theta) / theta_H.inner_product(theta_H)

In [ ]:
b = branching_rule("A4", "B2", "symmetric")
proj_theta, theta_H = projected_highest_root_from_branching_rule(b)
print(proj_theta)
print(theta_H)
%timeit embedding_index_from_branching_rule(b)

In [ ]:
b = branching_rule("A4", "B2", "symmetric")
proj_theta, theta_H = projected_highest_root_from_branching_rule(b)
print(proj_theta)
print(theta_H)
print(embedding_index_from_branching_rule(b))

In [ ]:
import io
import re
import contextlib

def projected_highest_root(b):
    """
    Compute the image of the highest root under a Sage branching rule
    using the simple-root restriction data.
    """

    source_type = b.Rtype()
    target_type = b.Stype()

    # Root systems
    G = RootSystem(source_type)
    H = RootSystem(target_type)

    G_root = G.root_lattice()
    H_amb  = H.ambient_space()

    # Highest root of source in simple-root basis
    theta = G_root.highest_root()
    theta_coeffs = theta.to_positive_combination()

    # Parse root restrictions from describe(verbose=True)
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        b.describe(verbose=True)
    txt = buf.getvalue()

    restrictions = {}

    for line in txt.splitlines():

        m = re.match(r"\s*(\d+)\s*=>\s*(\d+)\s*$", line)
        if m:
            i = int(m.group(1))
            j = int(m.group(2))

            e = [0]*H.rank()
            e[j-1] = 1

            restrictions[i] = H_amb(e)

    # Build projected highest root
    proj = H_amb.zero()

    for i, coeff in theta_coeffs.items():
        if i not in restrictions:
            raise ValueError(f"No restriction data for simple root {i}")

        proj += coeff * restrictions[i]

    return proj


def embedding_index_from_branching_rule(b):

    source_type = b.Rtype()
    target_type = b.Stype()

    H = RootSystem(target_type).ambient_space()

    proj_theta = projected_highest_root(b)
    theta_H = H.highest_root()

    return (
        proj_theta.inner_product(proj_theta)
        / theta_H.inner_product(theta_H)
    )

In [ ]:
import io
import re
import contextlib
from fractions import Fraction

def embedding_index_from_branching_rule(b):
    """
    Compute the Dynkin/embedding index from a Sage branching rule.

    Parameters
    ----------
    b : SageMath branching rule object

    Returns
    -------
    index : rational number
        ||image(highest_root_source)||^2 / ||highest_root_target||^2
    proj_highest_root : ambient element of the target root system
        The image of the source highest root in the target ambient space.
    """

    source_type = b.Rtype()
    target_type = b.Stype()

    target = RootSystem(target_type).ambient_space()
    target_highest = target.highest_root()

    # Capture b.describe(verbose=True), which Sage says includes the affine root
    # restriction. Since the affine root is -highest_root, negating that gives
    # the projected highest root. :contentReference[oaicite:1]{index=1}
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        b.describe(verbose=True)
    txt = buf.getvalue()

    m = re.search(r"^\s*0\s*=>\s*root\s*\(([^)]*)\)", txt, flags=re.M)
    if m is None:
        raise ValueError("Could not find the affine-root restriction in b.describe(verbose=True).")

    coords = [Fraction(s.strip()) for s in m.group(1).split(",")]
    if len(coords) != target.rank():
        raise ValueError(
            f"Parsed {len(coords)} coordinates, but target rank is {target.rank()}."
        )

    # describe() prints the restriction of the affine root α0 = -θ_source.
    # So the source highest root maps to the negative of that vector.
    proj_highest_root = target(tuple(-c for c in coords))

    index = proj_highest_root.inner_product(proj_highest_root) / target_highest.inner_product(target_highest)
    return index

In [ ]:
%timeit embedding_index(b)

In [ ]:
idx, proj = embedding_index_from_branching_rule("D4", "B3", "symmetric")
print(proj)
print(idx)